# UdaPlay — Part 2: Agent Implementation

Build a stateful AI research agent that answers natural language questions about video games.

## Workflow

```
User Query
    │
    ▼
[message_prep] ──► add system instructions + session history
    │
[llm_processor] ──► decide which tool to call
    │
    ├─ retrieve_game ──────► ChromaDB vector store
    │                              │
    ├─ evaluate_retrieval ◄────────┘   ──► LLM confidence check
    │         │
    │   confidence ≥ 0.7 ──────────────────────────────► answer
    │   confidence < 0.7 ──► game_web_search ──► Tavily ──► answer
    │
[tool_executor] ──► run selected tool, loop back to llm_processor
    │
[termination] ──► final answer + session memory updated
```

## Framework used

| Component | Purpose |
|-----------|--------|
| `lib.agents.Agent` | State machine + short-term session memory |
| `lib.tooling.tool` | Decorator that generates OpenAI tool schemas |
| `lib.llm.LLM` | OpenAI chat wrapper with tool support |
| `lib.vector_db.VectorStoreManager` | ChromaDB + OpenAI embeddings |
| `lib.state_machine.Run` | Execution object with full snapshot history |

In [1]:
# Only needed for Udacity workspace
import importlib.util
import sys

if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
from typing import List
from dotenv import load_dotenv

from lib.agents import Agent
from lib.llm import LLM
from lib.state_machine import Run
from lib.messages import BaseMessage
from lib.tooling import tool
from lib.vector_db import VectorStoreManager, CorpusLoaderService
from lib.rag import RAG

load_dotenv("config.env")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "https://openai.vocareum.com/v1")

assert OPENAI_API_KEY, "OPENAI_API_KEY must be set in config.env"
assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY must be set in config.env"

print("✓ All imports and environment ready")

✓ All imports and environment ready


## Load the Games RAG Pipeline

We reuse the same `VectorStoreManager` and `RAG` setup from Part 1.
The `games_rag` object is what the `retrieve_game` tool calls internally.

In [3]:
db = VectorStoreManager(OPENAI_API_KEY, api_base=OPENAI_BASE_URL)
loader_service = CorpusLoaderService(db)

games_store = loader_service.load_json(
    store_name="games",
    json_path="games.json",
)

rag_llm = LLM(model="gpt-4o-mini", temperature=0.3)
games_rag = RAG(llm=rag_llm, vector_store=games_store)

print("✓ Games RAG pipeline ready")

VectorStore `games` ready!


210 games from `games.json` added!
✓ Games RAG pipeline ready


## Tool Definitions

Each tool is decorated with `@tool`, which automatically builds the OpenAI function-calling
schema from the function's type hints and docstring — no manual JSON schema needed.

### Tool 1 — `retrieve_game`
Searches the internal ChromaDB vector store. Always called first.

In [4]:
@tool
def retrieve_game(query: str) -> str:
    """
    Search the internal game database for video game information.
    This is the PRIMARY knowledge source — ALWAYS call this tool first
    before attempting any other source.

    Source: Internal ChromaDB — 25 curated game records (2013–2023).

    args:
        query (str): Natural-language search query about a game, developer,
                     platform, genre, or release date.
    """
    result: Run = games_rag.invoke(query)
    final_state = result.get_final_state()
    
    # Include the raw retrieved documents alongside the generated answer
    # so evaluate_retrieval has enough context to judge quality.
    docs = final_state.get("documents", [])
    answer = final_state.get("answer", "No results found.")
    context = "\n\n".join(docs[:3]) if docs else "(no documents retrieved)"
    
    return (
        f"[Retrieved Documents]\n{context}\n\n"
        f"[Generated Answer]\n{answer}"
    )


print(f"✓ retrieve_game tool: {retrieve_game}")

✓ retrieve_game tool: <Tool name=retrieve_game params=['query']>


### Tool 2 — `evaluate_retrieval`
Uses a separate LLM call to assess whether the retrieved information actually answers the query.
Returns a JSON confidence score that drives the web-search decision.

In [5]:
from lib.messages import SystemMessage, UserMessage

_eval_llm = LLM(model="gpt-4o-mini", temperature=0.0)


@tool
def evaluate_retrieval(query: str, retrieved_info: str) -> str:
    """
    Evaluate whether the retrieved game information sufficiently answers
    the user's query. Returns a JSON confidence assessment.

    Call this IMMEDIATELY after retrieve_game to decide if web search
    is needed. If confidence_score < 0.7 or is_sufficient=false,
    call game_web_search next.

    args:
        query (str): The original user question.
        retrieved_info (str): The full output from retrieve_game.
    """
    prompt = f"""You are a quality evaluator for a gaming information retrieval system.

User query: {query}

Retrieved information:
{retrieved_info}

Assess whether the retrieved information sufficiently answers the query.
Respond ONLY with this JSON:
{{
  "confidence_score": <float 0.0-1.0>,
  "is_sufficient": <true if score >= 0.7, else false>,
  "reasoning": "<one sentence>",
  "missing_information": ["<item>"] or null
}}"""

    response = _eval_llm.invoke([
        SystemMessage(content="Respond only with valid JSON."),
        UserMessage(content=prompt),
    ])
    return response.content


print(f"✓ evaluate_retrieval tool: {evaluate_retrieval}")

✓ evaluate_retrieval tool: <Tool name=evaluate_retrieval params=['query', 'retrieved_info']>


### Tool 3 — `game_web_search`
Tavily web search fallback. Only called when `evaluate_retrieval` signals low confidence.

In [6]:
from tavily import TavilyClient

_tavily = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))


@tool
def game_web_search(query: str) -> str:
    """
    Search the web for video game information using the Tavily API.
    Use this tool ONLY when evaluate_retrieval returns confidence_score < 0.7
    or is_sufficient=false. Do NOT call this before evaluate_retrieval.

    Source: Live web search (Tavily), up-to-date as of today.

    args:
        query (str): Search query. Include the game title or developer name
                     for best results.
    """
    try:
        results = _tavily.search(
            query=f"{query} video game",
            search_depth="advanced",
            max_results=5,
        )
        parts = []
        for r in results.get("results", []):
            parts.append(
                f"Source: {r.get('url', 'N/A')}\n"
                f"Title: {r.get('title', 'N/A')}\n"
                f"Content: {r.get('content', '')}"
            )
        return "\n\n---\n\n".join(parts) if parts else "No web results found."
    except Exception as exc:
        return f"Web search failed: {exc}"


print(f"✓ game_web_search tool: {game_web_search}")

✓ game_web_search tool: <Tool name=game_web_search params=['query']>


## Build the Agent

The `Agent` class (from `lib.agents`) wires together:
- A `StateMachine`: `message_prep → llm_processor ↔ tool_executor → termination`
- `ShortTermMemory`: stores every `Run` per `session_id`, enabling multi-turn conversations

In [7]:
udaplay = Agent(
    model_name="gpt-4o-mini",
    temperature=0.3,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    instructions=(
        "You are UdaPlay, an expert AI research agent specializing in video game information. "
        "Follow this exact workflow for every query:\n"
        "1. ALWAYS call retrieve_game first to search the internal database.\n"
        "2. ALWAYS call evaluate_retrieval immediately after to assess the result.\n"
        "3. If confidence_score < 0.7 or is_sufficient=false, call game_web_search.\n"
        "4. After gathering information, provide a comprehensive, well-cited answer. "
        "Label each fact with its source: [Internal DB] or [Web Search].\n"
        "Use session context to resolve follow-up questions like 'What else did they make?'."
    ),
)

print("✓ UdaPlay agent ready")

✓ UdaPlay agent ready


## Helper — display messages

In [8]:
def print_messages(messages: List[BaseMessage]) -> None:
    """Pretty-print the message trace from a Run."""
    for m in messages:
        tc = getattr(m, "tool_calls", None)
        tc_str = f", tool_calls={[c.function.name for c in tc]}" if tc else ""
        print(f"  → (role={m.role}{tc_str})")
        if m.content:
            # Truncate long content for readability
            preview = m.content[:300].replace("\n", " ")
            print(f"    {preview}{'...' if len(m.content) > 300 else ''}")

---
## Example Query 1 — Developer lookup

Expected path: `retrieve_game → evaluate_retrieval → answer`  
Expected source: Internal DB (FIFA 21 is in the dataset)

In [9]:
run_1 = udaplay.invoke(
    query="Who developed FIFA 21?",
    session_id="developer_query",
)

print("\n=== Final Answer ===")
final_1 = run_1.get_final_state()["messages"][-1].content
print(final_1)

print("\n=== Message Trace ===")
print_messages(run_1.get_final_state()["messages"])

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


[StateMachine] Executing step: llm_processor
[StateMachine] Starting: __entry__


[StateMachine] Executing step: retrieve
[StateMachine] Executing step: augment


[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__
[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

=== Final Answer ===
FIFA 21 was developed by EA Sports [Internal DB].

=== Message Trace ===
  → (role=system)
    You are UdaPlay, an expert AI research agent specializing in video game information. Follow this exact workflow for every query: 1. ALWAYS call retrieve_game first to search the internal database. 2. ALWAYS call evaluate_retrieval immediately after to assess the result. 3. If confidence_score < 0.7 ...
  → (role=user)
    Who developed FIFA 21?
  → (role=assistant, tool_calls=['retrieve_game'])
  → (role=tool)
    "[Retrieved Documents]\nTitle: FIFA 21\nDeveloper: EA Sports\nPublisher: Electronic Arts\nRelease Date: 2020-10-09\nPlatforms: PlayStation 4, Xbox One, PC, Nintendo Switch, PlayStation 5, Xbox Series X/S\nGenre: Sports\nDescription: FIFA 21 is a football simulation video game published by Electronic...
  → (role=assistant, tool_calls=['evaluate_retrieval'])
  → (role=tool)


## Example Query 2 — Release date and platforms

Expected path: `retrieve_game → evaluate_retrieval → answer`  
Expected source: Internal DB

In [10]:
run_2 = udaplay.invoke(
    query="When was God of War Ragnarok released and on what platforms?",
    session_id="release_query",
)

print("\n=== Final Answer ===")
print(run_2.get_final_state()["messages"][-1].content)

print("\n=== Message Trace ===")
print_messages(run_2.get_final_state()["messages"])

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


[StateMachine] Executing step: llm_processor
[StateMachine] Starting: __entry__


[StateMachine] Executing step: retrieve
[StateMachine] Executing step: augment


[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__
[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

=== Final Answer ===
God of War Ragnarok was released on November 9, 2022. It is available on PlayStation 5 and PlayStation 4 [Internal DB].

=== Message Trace ===
  → (role=system)
    You are UdaPlay, an expert AI research agent specializing in video game information. Follow this exact workflow for every query: 1. ALWAYS call retrieve_game first to search the internal database. 2. ALWAYS call evaluate_retrieval immediately after to assess the result. 3. If confidence_score < 0.7 ...
  → (role=user)
    When was God of War Ragnarok released and on what platforms?
  → (role=assistant, tool_calls=['retrieve_game'])
  → (role=tool)
    "[Retrieved Documents]\nTitle: God of War Ragnarok\nDeveloper: Santa Monica Studio\nPublisher: Sony Interactive Entertainment\nRelease Date: 2022-11-09\nPlatforms: PlayStation 5, PlayStation 4\nGenre: Action-Adventure\nDescription: God of War Ragnarok is an action-adv

## Example Query 3 — Classic platform launch

Expected path: `retrieve_game → evaluate_retrieval → answer`  
Expected source: Internal DB (Pokémon Red is in the dataset)

In [11]:
run_3 = udaplay.invoke(
    query="What platform was Pokémon Red originally launched on?",
    session_id="platform_query",
)

print("\n=== Final Answer ===")
print(run_3.get_final_state()["messages"][-1].content)

print("\n=== Message Trace ===")
print_messages(run_3.get_final_state()["messages"])

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


[StateMachine] Executing step: llm_processor
[StateMachine] Starting: __entry__


[StateMachine] Executing step: retrieve
[StateMachine] Executing step: augment


[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__
[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

=== Final Answer ===
Pokémon Red was originally launched on the Game Boy on February 27, 1996. This game, developed by Game Freak and published by Nintendo, was the first entry in the Pokémon main series, allowing players to catch, train, and battle various Pokémon [Internal DB].

=== Message Trace ===
  → (role=system)
    You are UdaPlay, an expert AI research agent specializing in video game information. Follow this exact workflow for every query: 1. ALWAYS call retrieve_game first to search the internal database. 2. ALWAYS call evaluate_retrieval immediately after to assess the result. 3. If confidence_score < 0.7 ...
  → (role=user)
    What platform was Pokémon Red originally launched on?
  → (role=assistant, tool_calls=['retrieve_game'])
  → (role=tool)
    "[Retrieved Documents]\nTitle: Pokemon Red\nDeveloper: Game Freak\nPublisher: Nintendo\nRelease Date: 1996-02-27\nPlatforms: Game Boy\n

## Example Query 4 — Current events (triggers web search)

The internal database only contains games up to 2023. Questions about
*current* or upcoming projects should score low confidence, triggering Tavily.

Expected path: `retrieve_game → evaluate_retrieval → game_web_search → answer`  
Expected sources: Internal DB + Web Search

In [12]:
run_4 = udaplay.invoke(
    query="What is Rockstar Games currently working on?",
    session_id="rockstar_query",
)

print("\n=== Final Answer ===")
print(run_4.get_final_state()["messages"][-1].content)

print("\n=== Message Trace ===")
print_messages(run_4.get_final_state()["messages"])

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


[StateMachine] Executing step: llm_processor
[StateMachine] Starting: __entry__


[StateMachine] Executing step: retrieve
[StateMachine] Executing step: augment


[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__
[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

=== Final Answer ===
As of 2023, Rockstar Games is reportedly working on several exciting projects:

1. **Grand Theft Auto VI (GTA 6)**: This is perhaps the most anticipated title, with Rockstar gearing up for its release on platforms like PlayStation 5, Xbox Series X, and PC. The game is expected to expand the GTA universe significantly and is one of the main focuses of the studio's current development efforts [Web Search].

2. **Max Payne Remakes**: Rockstar is collaborating with Remedy Entertainment to remake the classic Max Payne games. This project is described as a flagship production, aiming to modernize the original games while retaining their core essence. The remakes are currently in full production, targeting a release window around 2026 [Web Search].

3. **GTA San Andreas VR**: There are plans for a virtual reality version of GTA San Andreas, which would bring the classic game into a n

## Example Query 5 — Out-of-dataset title

A game not in the local dataset forces the web fallback.

Expected path: `retrieve_game → evaluate_retrieval → game_web_search → answer`

In [13]:
run_5 = udaplay.invoke(
    query="Tell me about Metaphor: ReFantazio — who made it and when was it released?",
    session_id="metaphor_query",
)

print("\n=== Final Answer ===")
print(run_5.get_final_state()["messages"][-1].content)

print("\n=== Message Trace ===")
print_messages(run_5.get_final_state()["messages"])

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


[StateMachine] Executing step: llm_processor
[StateMachine] Starting: __entry__


[StateMachine] Executing step: retrieve
[StateMachine] Executing step: augment


[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__
[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

=== Final Answer ===
**Metaphor: ReFantazio** is a role-playing video game developed by **Studio Zero** and published by **Sega**. It is set to be released on **October 11, 2024** for platforms including PlayStation 4, PlayStation 5, Windows, and Xbox Series X/S. Additionally, a version for the Nintendo Switch 2 is planned for release on **November 12, 2026**. The game was first announced under the working title "Project Re:Fantasy" in December 2016, with significant details revealed only in 2023 [Web Search].

The game features a medieval European-inspired fantasy setting and follows a non-silent protagonist on a quest to protect their kingdom, forging alliances along the way [Web Search].

=== Message Trace ===
  → (role=system)
    You are UdaPlay, an expert AI research agent specializing in video game information. Follow this exact workflow for every query: 1. ALWAYS call retrieve_game first t

---
## Session Memory — Multi-turn Conversation

The `Agent` uses `ShortTermMemory` to persist all `Run` objects per `session_id`.
When the same `session_id` is reused, the previous message history is injected
so follow-up questions resolve correctly.

In [14]:
# Turn 1 — establish a topic
session = "cd_projekt_session"

run_a = udaplay.invoke(
    query="What games has CD Projekt Red developed?",
    session_id=session,
)
print("=== Turn 1 ===")
print(run_a.get_final_state()["messages"][-1].content)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


[StateMachine] Executing step: llm_processor
[StateMachine] Starting: __entry__


[StateMachine] Executing step: retrieve
[StateMachine] Executing step: augment


[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__
[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
=== Turn 1 ===
CD Projekt Red has developed the following notable games:

1. **The Witcher** (2007)
   - **Platforms**: PC, Mac
   - **Genre**: Action RPG
   - **Description**: This was CD Projekt Red's debut game, adapting Andrzej Sapkowski's Polish fantasy novels. It features a morally complex narrative centered around Geralt of Rivia and includes unique gameplay elements like a dice poker minigame and a romance card system.

2. **The Witcher 3: Wild Hunt** (2015)
   - **Platforms**: PC, PlayStation 4, Xbox One, Nintendo Switch, PlayStation 5, Xbox Series X/S
   - **Genre**: Action RPG
   - **Description**: A critically acclaimed sequel that follows Geralt as he searches for his adopted daughter Ciri in a vast open world filled with complex moral choices and engaging storylines.

3. **Cyberpunk 2077** (2020)
   - **Platforms**: PC, PlayStation 4, Xbox One, PlayStation 5, Xbox Series X/S, Google S

In [15]:
# Turn 2 — follow-up uses pronoun; agent resolves "their" from session history
run_b = udaplay.invoke(
    query="Which of their games has the most expansions or DLC?",
    session_id=session,
)
print("=== Turn 2 ===")
print(run_b.get_final_state()["messages"][-1].content)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


[StateMachine] Executing step: llm_processor
[StateMachine] Starting: __entry__


[StateMachine] Executing step: retrieve
[StateMachine] Executing step: augment


[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__
[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
=== Turn 2 ===
CD Projekt Red's game with the most expansions and downloadable content (DLC) is **The Witcher 3: Wild Hunt**. Here are the details:

1. **Expansions**:
   - **Hearts of Stone**: Released on October 13, 2015, this expansion adds a new storyline and quests.
   - **Blood and Wine**: Released on May 31, 2016, this expansion introduces a new region called Toussaint and is significantly larger than Hearts of Stone.

2. **DLC**: The Witcher 3 also features **16 free DLC packs**, which include various cosmetic items, quests, and gameplay enhancements. These were released in the lead-up to and following the game's launch.

In total, **The Witcher 3: Wild Hunt** has two major expansions and 16 pieces of free DLC, making it the CD Projekt Red game with the most additional content. In contrast, **Cyberpunk 2077** has only one expansion, titled **Phantom Liberty**, released on September 26, 2023

In [16]:
# Inspect the full session run history
all_runs = udaplay.get_session_runs(session)
print(f"Session '{session}' has {len(all_runs)} run(s):")
for i, run in enumerate(all_runs, 1):
    msgs = run.get_final_state()["messages"]
    user_msgs = [m for m in msgs if m.role == "user"]
    print(f"  Run {i}: {len(msgs)} messages, user query: '{user_msgs[-1].content[:60]}...'")

Session 'cd_projekt_session' has 2 run(s):
  Run 1: 7 messages, user query: 'What games has CD Projekt Red developed?...'
  Run 2: 15 messages, user query: 'Which of their games has the most expansions or DLC?...'


## Interactive Mode

Change `question` to ask anything about video games.

In [17]:
question = "Which games in the database were released on Nintendo Switch?"

run = udaplay.invoke(query=question, session_id="interactive")
print(run.get_final_state()["messages"][-1].content)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


[StateMachine] Executing step: llm_processor
[StateMachine] Starting: __entry__


[StateMachine] Executing step: retrieve
[StateMachine] Executing step: augment


[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__
[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Here are some notable games that were released on the Nintendo Switch:

1. **Pokemon Sword and Shield**
   - **Developer:** Game Freak
   - **Publisher:** Nintendo / The Pokemon Company
   - **Release Date:** November 15, 2019
   - **Genre:** Role-Playing Game
   - **Description:** This game brought the mainline Pokémon series to the Nintendo Switch for the first time, set in the UK-inspired Galar region with 81 new Pokémon. It introduced an open-world section called the Wild Area and sold over 25 million copies despite some controversy regarding the limited National Dex. [Internal DB]

2. **Mario Kart 8 Deluxe**
   - **Developer:** Nintendo EPD
   - **Publisher:** Nintendo
   - **Release Date:** April 28, 2017
   - **Genre:** Racing
   - **Description:** An enhanced port of Mario Kart 8, this game features all DLC content, a revamped battle mode, and new characters. It is the best-selling Switch g